# Indicadores – Pesca e Aquicultura (RESEX/UCs)

Execute as células em ordem (`Shift+Enter`) ou use **Executar Tudo**.
Só a célula 1 precisa ser editada: nomes dos arquivos e critérios.

## 1. Configuração (edite aqui)

In [90]:
# Arquivo na mesma pasta deste caderno (ou use o caminho completo)
ARQUIVO = "consolidado_dasdosGerais_181126.xlsx"                    # seu arquivo com as 3 abas
ABA_DADOS_GERAIS = "Dados Gerais"          # use None se não quiser usar esta aba
ABA_MEMBROS      = "Membros Familiares"
ABA_ATIVIDADES   = "Atividades Produtivas"
# (maiúsculas, acentos e singular/plural nos nomes das abas não importam)

# Critérios da aquicultura (coluna "Produto Atividade Matéria Prima")
PRODUTOS_AQUICULTURA = ["Peixes (piscicultura)", "Camarões (carcinicultura)", "Ostras (ostreicultura)" ]
# Texto procurado em "Outro Produto" (não diferencia maiúsculas; acha em qualquer posição)
TEXTOS_OUTRO_AQUICULTURA = ["PSICULTURA", "VIVEIRO DE TAMBAQUI"]

# Nomes das colunas (só mude se os cabeçalhos forem diferentes)
FAM, INTEG, SEXO, RGP, OCUP, NASC = "Código Familiar", "Código Integrante", "Sexo", "RGP - Possui", "Principais Ocupações", "Data Nascimento"
ATV, PROD, OUTRO = "Principais Atividades Produtivas", "Produto Atividade Matéria Prima", "Outro Produto"
IDADE_MINIMA = 18
DATA_REFERENCIA = None

## 2. Leitura e limpeza dos dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import unicodedata

# texto -> sem acentos, em letras minúsculas, sem espaços no início ou no fim, sem espaços repetidos
def sanitize_text(text):
    text_without_accents = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return " ".join(text_without_accents.lower().split())

# nomes das colunas provenientes da configuração, no mesmo formato dos cabeçalhos sanitizados
FAM, INTEG, SEXO, RGP, OCUP, ATV, PROD, OUTRO, NASC = [
    sanitize_text(column_name)
    for column_name in (FAM, INTEG, SEXO, RGP, OCUP, ATV, PROD, OUTRO, NASC)
]

# o "calamine" lê arquivos .xlsx muito mais rapidamente; caso não seja possível, utiliza o mecanismo padrão
try:
    excel_file = pd.ExcelFile(ARQUIVO, engine="calamine")
except (ImportError, ValueError):
    excel_file = pd.ExcelFile(ARQUIVO)
print("Sheets found:", excel_file.sheet_names)

# {nome da aba higienizado: nome da aba original}, criado uma única vez
sheet_names_by_sanitized = {
    sanitize_text(sheet_name): sheet_name for sheet_name in excel_file.sheet_names
}

def find_sheet_name(requested_sheet_name):
    sanitized_name = sanitize_text(requested_sheet_name)
    assert sanitized_name in sheet_names_by_sanitized, (
        f"Sheet '{requested_sheet_name}' not found. Sheets in file: {excel_file.sheet_names}"
    )
    return sheet_names_by_sanitized[sanitized_name]

# lê APENAS as colunas necessárias de uma aba, com cabeçalhos sanitizados
def read_sheet(requested_sheet_name, needed_columns):
    table = pd.read_excel(
        excel_file,
        sheet_name=find_sheet_name(requested_sheet_name),
        usecols=lambda column: sanitize_text(column) in needed_columns,
    )
    table.columns = [sanitize_text(column) for column in table.columns]
    assert not table.columns.duplicated().any(), f"Duplicated headers in sheet '{requested_sheet_name}'"
    missing_columns = [column for column in needed_columns if column not in table.columns]
    assert not missing_columns, f"Columns not found in '{requested_sheet_name}': {missing_columns}"
    return table

# remove espaços nas extremidades e reduz espaços repetidos; opera apenas sobre os valores únicos
def clean_spaces(table, columns):
    for column in columns:
        values = table[column]
        cleaned_by_value = {
            value: " ".join(str(value).split()) for value in values.dropna().unique()
        }
        table[column] = values.map(cleaned_by_value)   # células vazias permanecem vazias
    return table

gerais = clean_spaces(read_sheet(ABA_DADOS_GERAIS, [FAM]), [FAM])
ativ = clean_spaces(read_sheet(ABA_ATIVIDADES, [FAM, ATV, PROD, OUTRO]), [FAM, ATV, PROD, OUTRO])
memb = clean_spaces(read_sheet(ABA_MEMBROS, [FAM, INTEG, SEXO, RGP, OCUP, NASC]), [FAM, INTEG, SEXO, RGP, OCUP])

gerais = gerais.drop_duplicates()

# idade em anos completos, na data de referência
reference_date = pd.Timestamp(DATA_REFERENCIA) if DATA_REFERENCIA else pd.Timestamp.today().normalize()
# transformacao de data de nascimento para o pandas
birth_date = pd.to_datetime(memb[NASC], dayfirst=True, errors="coerce", format="mixed")
birthday_not_yet = (
    (birth_date.dt.month > reference_date.month)
    | ((birth_date.dt.month == reference_date.month) & (birth_date.dt.day > reference_date.day))
).astype(int)

memb["age"] = reference_date.year - birth_date.dt.year - birthday_not_yet
memb["is_adult"] = memb["age"] >= IDADE_MINIMA
 
# busca um texto dentro de uma coluna (case-insensitive)
def contains_text(text_column, searched_text):
    return text_column.fillna("").astype(str).str.contains(searched_text, case=False, regex=False)

print(f"Atividade Produtiva: {len(ativ):,} linha | {ativ[FAM].nunique():,} famílias")
print(f"Membros familiares:    {len(memb):,} pessoas | {memb[FAM].nunique():,} famílias")
print(f"Dados Gerais: {gerais[FAM].nunique():,} famílias")

## 3. Pesca

In [ ]:
fam_pesca = set(ativ.loc[contains_text(ativ[ATV], "Atividade pesqueira"), FAM])

print(f"Famílias com atividade pesqueira:       {len(fam_pesca):,}")

# ---------- PESSOAS (aba Membros Familiares) ----------
# Pessoas com "Atividade pesqueira" na coluna Principais Ocupações (mesmo com outras ocupações juntas)
# pesca_pessoas = memb[contains_text(memb[OCUP], "Atividade pesqueira")]
pesca_pessoas = memb[contains_text(memb[OCUP], "Atividade pesqueira") & memb["is_adult"]]

total_pesca = len(pesca_pessoas)

cont = pesca_pessoas[RGP].fillna("(em branco)").value_counts()
pesca_rgp_sim   = int(cont.get("Sim", 0))
pesca_rgp_nao   = int(cont.get("Não", 0))
pesca_rgp_nsi   = int(cont.get("Não sabe/Não informou", 0))
pesca_rgp_vazio = int(cont.get("(em branco)", 0))

print(f"\nAdultos com atividade pesqueira: {total_pesca:,}")
print(f"  RGP Sim:                 {pesca_rgp_sim:,}  ({pesca_rgp_sim/total_pesca:.1%})")
print(f"  RGP Não:                 {pesca_rgp_nao:,}  ({pesca_rgp_nao/total_pesca:.1%})")
print(f"  Não sabe/Não informou:   {pesca_rgp_nsi:,}  ({pesca_rgp_nsi/total_pesca:.1%})")
if pesca_rgp_vazio:
    print(f"  Em branco:               {pesca_rgp_vazio:,}  ({pesca_rgp_vazio/total_pesca:.1%})")

# Tabela por sexo x RGP
tab_pesca = pd.crosstab(pesca_pessoas[SEXO], pesca_pessoas[RGP], margins=True, margins_name="Total")
tab_pesca

## 4. Aquicultura

In [ ]:
# ---------- 1) FAMÍLIAS: aba Atividades Produtivas ----------
d = ativ[PROD].fillna("")

# a) produto na lista de aquicultura
cond_produto = d.isin(PRODUTOS_AQUICULTURA)

# b) produto = "Outro" E "Outro Produto" com um dos textos
achou_texto = pd.concat([contains_text(ativ[OUTRO], t) for t in TEXTOS_OUTRO_AQUICULTURA], axis=1).any(axis=1)
cond_outro = (d == "Outro") & achou_texto

fam_aqui = set(ativ.loc[cond_produto | cond_outro, FAM])      # códigos familiares únicos

print(f"Famílias com aquicultura: {len(fam_aqui):,}")

# ---------- 2) VERIFICAÇÃO: aba Membros Familiares ----------
# todas as pessoas cujo código familiar está na lista acima
# memb_aqui = memb[memb[FAM].isin(fam_aqui)]
memb_aqui = memb[memb[FAM].isin(fam_aqui) & memb["is_adult"]]
total_aqui = len(memb_aqui)

cont = memb_aqui[RGP].fillna("(em branco)").value_counts()
aqui_rgp_sim   = int(cont.get("Sim", 0))
aqui_rgp_nao   = int(cont.get("Não", 0))
aqui_rgp_nsi   = int(cont.get("Não sabe/Não informou", 0))
aqui_rgp_vazio = int(cont.get("(em branco)", 0))

print(f"\nAdultos, nessas famílias, na aquicultura: {total_aqui:,}")
if total_aqui:
    print(f"  RGP Sim:                 {aqui_rgp_sim:,}  ({aqui_rgp_sim/total_aqui:.1%})")
    print(f"  RGP Não:                 {aqui_rgp_nao:,}  ({aqui_rgp_nao/total_aqui:.1%})")
    print(f"  Não sabe/Não informou:   {aqui_rgp_nsi:,}  ({aqui_rgp_nsi/total_aqui:.1%})")
    if aqui_rgp_vazio:
        print(f"  Em branco:               {aqui_rgp_vazio:,}  ({aqui_rgp_vazio/total_aqui:.1%})")

tab_aqui = pd.crosstab(memb_aqui[SEXO], memb_aqui[RGP], margins=True, margins_name="Total")
tab_aqui

## 5. Quadro-resumo

In [ ]:
# conta as linhas de uma tabela de pessoas, opcionalmente filtradas por sexo e/ou RGP
def count_people(people, sex=None, rgp=None):
    selected = pd.Series(True, index=people.index)
    if sex:
        selected &= people[SEXO] == sex
    if rgp:
        selected &= people[RGP] == rgp
    return int(selected.sum())

summary_rows = [
    ("Total de famílias", gerais[FAM].nunique()),

    ("Famílias com Atividade pesqueira como atividade principal", len(fam_pesca)),
    ("Familiares com RGP (Sim) - Atividade pesqueira", count_people(pesca_pessoas, rgp="Sim")),
    ("Familiares sem RGP (Não) - Atividade pesqueira", count_people(pesca_pessoas, rgp="Não")),
    ("Mulheres na pesca", count_people(pesca_pessoas, "Feminino")),
    ("Mulheres na pesca com RGP", count_people(pesca_pessoas, "Feminino", "Sim")),
    ("Homens na pesca", count_people(pesca_pessoas, "Masculino")),
    ("Homens na pesca com RGP", count_people(pesca_pessoas, "Masculino", "Sim")),

    ("Famílias com Aquicultura como atividade principal", len(fam_aqui)),
    ("Familiares com RGP (Sim) - Aquicultura", count_people(memb_aqui, rgp="Sim")),
    ("Familiares sem RGP (Não) - Aquicultura", count_people(memb_aqui, rgp="Não")),
    ("Mulheres na aquicultura", count_people(memb_aqui, "Feminino")),
    ("Mulheres na aquicultura com RGP", count_people(memb_aqui, "Feminino", "Sim")),
    ("Homens na aquicultura", count_people(memb_aqui, "Masculino")),
    ("Homens na aquicultura com RGP", count_people(memb_aqui, "Masculino", "Sim")),
]
resumo = pd.DataFrame(summary_rows, columns=["Indicador", "Quantitativo"])
resumo

## 6. Gráficos

In [ ]:
cores = {"Sim": "#2a9d8f", "Não": "#9aa5b1", "Não sabe/Não informou": "#e9c46a"}
sexos = ["Feminino", "Masculino"]

# 6.1 Famílias por atividade principal
por_atv = ativ.groupby(ATV)[FAM].nunique().sort_values()
por_atv.index = [i.split(" (")[0][:60] for i in por_atv.index]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(por_atv.index, por_atv.values, color="#264653")
ax.set_title("Famílias por atividade produtiva principal")
ax.set_xlabel("Nº de famílias")
for i, v in enumerate(por_atv.values): ax.text(v, i, f" {v:,}", va="center", fontsize=8)
plt.tight_layout(); plt.savefig("grafico_atividades.png", dpi=150); plt.show()

# 6.2 / 6.3 Pessoas por sexo e RGP (pesca e aquicultura)
def barras_sexo_rgp(df, titulo, arq):
    t = pd.crosstab(df[SEXO], df[RGP]).reindex(sexos).fillna(0)
    fig, ax = plt.subplots(figsize=(6, 4))
    base = pd.Series(0, index=t.index)
    for col in [c for c in cores if c in t.columns]:
        ax.bar(t.index, t[col], bottom=base, label=f"RGP: {col}", color=cores[col])
        base += t[col]
    for i, tot in enumerate(base): ax.text(i, tot, f"{int(tot):,}", ha="center", va="bottom")
    ax.set_title(titulo); ax.set_ylabel("Pessoas"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(arq, dpi=150); plt.show()

barras_sexo_rgp(pesca_pessoas, "Pesca: pessoas por sexo e RGP", "grafico_pesca_sexo_rgp.png")
barras_sexo_rgp(memb_aqui, "Aquicultura: familiares por sexo e RGP", "grafico_aquicultura_sexo_rgp.png")

## 7. Exportar resultados para Excel

In [ ]:
with pd.ExcelWriter("resultados_indicadores.xlsx") as xw:
    resumo.to_excel(xw, sheet_name="Resumo", index=False)
    tab_pesca.to_excel(xw, sheet_name="Pesca sexo x RGP")
    tab_aqui.to_excel(xw, sheet_name="Aquicultura sexo x RGP")
print("Arquivo gerado: resultados_indicadores.xlsx (e os gráficos .png)")